[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_12/MFM_ch12_seminar/MFM_ch12_seminar.ipynb)

# MFM_ch12_seminar

Seminar 12 computations: convexity of Bitcoin call prices in the strike, the implied forward from parity regressions, convergence of the binomial tree, Black-Scholes Greeks, delta-gamma hedging, implied volatility by Newton, truncation and discretisation of a variance strip, hedging error with bootstrap intervals, SVI fit of a Bitcoin smile, variance risk premium of the S&P 500 and Bitcoin with Newey-West errors, Mincer-Zarnowitz regression, a risk-neutral density with bootstrap bands and a synthetic variance swap strategy.

*Modelling Financial Markets (MFM), Chapter 12: Options and the Volatility Surface. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_12'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import stats, optimize, integrate
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

N, n = stats.norm.cdf, stats.norm.pdf

## Style and helpers

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'   # doar linii de referinta, benzi, grila
LightGray = '#DADADA'
Teal     = '#17A2B8'

HERE = '.'
SEED = 42
BASE = {'S': 100.0, 'K': 100.0, 'T': 0.25, 'r': 0.04, 'sigma': 0.2}
HEDGE = {'S0': 100.0, 'K': 100.0, 'T': 0.25, 'r': 0.04, 'sigma': 0.2, 'mu': 0.08, 'nstep': 252, 'npath': 20000}


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def fig_legend_bottom(fig, handles, ncol=3, y=0.0):
    """O singura legenda pentru o figura cu mai multe panouri, sub figura."""
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def nw_mean(x, lags):
    """Media si eroarea standard Newey-West (HAC) a unei serii."""
    x = np.asarray(x, float)
    m = sm.OLS(x, np.ones_like(x)).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    return float(m.params[0]), float(m.bse[0])


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, np.ndarray)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, pd.Timestamp):
        return str(o)
    return o

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# copia locala a folderului data/market, daca notebook-ul ruleaza din repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
SNAP_DIR = next((d for d in ('.', '../Quantlets/Ch_12', '../../Quantlets/Ch_12')
                 if os.path.exists(os.path.join(d, 'ch12_deribit_snapshot.csv'))), '.')
END = '2026-09-18'

# nume -> (simbol, coloana, tip)
SERIES = {
    'sp500': ('GSPC.INDX', 'close', 'index'),
    'spy':   ('SPY.US', 'adjusted_close', 'etf'),
    'btc':   ('BTC-USD.CC', 'close', 'crypto'),
    'vix':   ('VIX.INDX', 'close', 'vol'),
    'vix9d': ('VIX9D.INDX', 'close', 'vol'),
    'vix3m': ('VIX3M.INDX', 'close', 'vol'),
    'vvix':  ('VVIX.INDX', 'close', 'vol'),
}
DERIBIT = 'https://www.deribit.com/api/v2/public/'
SNAP_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/Quantlets/Ch_12/ch12_deribit_snapshot.csv'


def read_market(symbol):
    """Citeste data/market/<SIMBOL>.csv local sau din repo-ul GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def load_close(name, start=None, end=END):
    """Pretul (sau nivelul indicelui) zilnic; indicii: doar zilele lucratoare, fara zilele cu valoare neschimbata."""
    symbol, col, kind = SERIES[name]
    s = read_market(symbol)[col].loc[start:end]
    s = s[s > 0].dropna()
    if kind != 'crypto':
        s = s[s.index.dayofweek < 5]
    if kind == 'index':
        s = s[s.diff() != 0]
    return s.rename(name)


def log_returns(name, start=None, end=END):
    """Randamente log pe calendarul propriu al seriei."""
    return np.log(load_close(name, start, end)).diff().dropna().rename(name)


def spy_open_close(start=None, end=END):
    """Deschiderea si inchiderea zilnica SPY (preturi neajustate: raportul din aceeasi zi nu depinde de ajustari)."""
    d = read_market('SPY.US').loc[start:end, ['open', 'close']]
    d = d[(d > 0).all(axis=1)]
    return d[d.index.dayofweek < 5]


def _get(method, **params):
    url = DERIBIT + method + '?' + '&'.join(f'{k}={v}' for k, v in params.items())
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    return json.load(urllib.request.urlopen(req, timeout=60))['result']


def deribit_chain(path=None, refresh=False):
    """Lantul de optiuni BTC (preturi de marcaj, volatilitati implicite) la momentul instantaneului."""
    path = path or os.path.join(SNAP_DIR, 'ch12_deribit_snapshot.csv')
    if not refresh:
        for src in (path, SNAP_RAW):
            try:
                return pd.read_csv(src, parse_dates=['snapshot_utc', 'expiry'])
            except Exception:
                pass
    rows = _get('get_book_summary_by_currency', currency='BTC', kind='option')
    t0 = pd.Timestamp.now(tz='UTC').floor('min').tz_localize(None)
    out = []
    for x in rows:
        _, exp, strike, cp = x['instrument_name'].split('-')
        out.append(dict(snapshot_utc=t0, instrument=x['instrument_name'],
                        expiry=pd.to_datetime(exp, format='%d%b%y') + pd.Timedelta(hours=8),
                        strike=float(strike), type='call' if cp == 'C' else 'put',
                        mark_iv=x['mark_iv'], mark_price=x['mark_price'], bid=x['bid_price'], ask=x['ask_price'],
                        forward=x['underlying_price'], index=x['estimated_delivery_price'],
                        open_interest=x['open_interest'], volume_usd=x['volume_usd']))
    d = pd.DataFrame(out).sort_values(['expiry', 'strike', 'type']).reset_index(drop=True)
    d.to_csv(path, index=False)
    return d


def dvol_history(start='2021-03-24', end=END):
    """Indicele DVOL (volatilitatea implicita pe 30 de zile a optiunilor BTC, anualizata, in %), valori zilnice."""
    t1 = int(pd.Timestamp(end).tz_localize('UTC').timestamp() * 1000) + 86_400_000
    t0 = int(pd.Timestamp(start).tz_localize('UTC').timestamp() * 1000)
    data = []
    while True:
        r = _get('get_volatility_index_data', currency='BTC', start_timestamp=t0, end_timestamp=t1, resolution='1D')
        data += r['data']
        if not r.get('continuation') or r['continuation'] <= t0:
            break
        t1 = r['continuation']
    d = pd.DataFrame(data, columns=['t', 'open', 'high', 'low', 'close']).drop_duplicates('t')
    d.index = pd.to_datetime(d['t'], unit='ms').dt.normalize()
    return d.sort_index()['close'].loc[start:end].rename('dvol')

## Option pricing tools

In [ ]:
def bs_price(S, K, T, r, sigma, kind='call', q=0.0):
    """Pretul Black-Scholes al unei optiuni europene."""
    S, K, T, sigma = map(np.asarray, (S, K, T, sigma))
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if kind == 'call':
        return S * np.exp(-q * T) * N(d1) - K * np.exp(-r * T) * N(d2)
    return K * np.exp(-r * T) * N(-d2) - S * np.exp(-q * T) * N(-d1)


def bs_greeks(S, K, T, r, sigma, kind='call', q=0.0):
    """Delta, gamma, vega (la 1 punct de volatilitate), theta (pe zi calendaristica), rho; plus d1, d2."""
    S, K, T, sigma = map(np.asarray, (S, K, T, sigma))
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    gamma = np.exp(-q * T) * n(d1) / (S * sigma * np.sqrt(T))
    vega = S * np.exp(-q * T) * n(d1) * np.sqrt(T) / 100
    common = -S * np.exp(-q * T) * n(d1) * sigma / (2 * np.sqrt(T))
    if kind == 'call':
        delta = np.exp(-q * T) * N(d1)
        theta = common - r * K * np.exp(-r * T) * N(d2) + q * S * np.exp(-q * T) * N(d1)
        rho = K * T * np.exp(-r * T) * N(d2) / 100
    else:
        delta = -np.exp(-q * T) * N(-d1)
        theta = common + r * K * np.exp(-r * T) * N(-d2) - q * S * np.exp(-q * T) * N(-d1)
        rho = -K * T * np.exp(-r * T) * N(-d2) / 100
    return dict(d1=d1, d2=d2, delta=delta, gamma=gamma, vega=vega, theta=theta / 365, rho=rho)


def implied_vol(price, S, K, T, r, kind='call', q=0.0, lo=1e-4, hi=5.0):
    """Volatilitatea implicita: solutia unica a ecuatiei BS(sigma) = pret (vega > 0)."""
    f = lambda s: bs_price(S, K, T, r, s, kind, q) - price
    if f(lo) > 0 or f(hi) < 0:
        return np.nan
    return optimize.brentq(f, lo, hi, xtol=1e-10)


def newton_iv(price, S, K, T, r, sigma0=0.2, kind='call', steps=4):
    """Pasii metodei Newton: sigma_{k+1} = sigma_k - (BS(sigma_k) - pret) / vega(sigma_k)."""
    out, s = [], sigma0
    for _ in range(steps):
        p = float(bs_price(S, K, T, r, s, kind))
        v = float(bs_greeks(S, K, T, r, s, kind)['vega']) * 100
        s_new = s - (p - price) / v
        out.append(dict(sigma=s, price=p, vega=v, sigma_next=s_new))
        s = s_new
    return out


def crr_price(S, K, T, r, sigma, steps, kind='call', american=False):
    """Arborele binomial Cox-Ross-Rubinstein: u = exp(sigma sqrt(dt)), d = 1/u, p = (e^{r dt} - d)/(u - d)."""
    dt = T / steps
    u = np.exp(sigma * np.sqrt(dt)); d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)
    j = np.arange(steps + 1)
    ST = S * u ** (steps - j) * d ** j
    V = np.maximum(ST - K, 0) if kind == 'call' else np.maximum(K - ST, 0)
    for i in range(steps - 1, -1, -1):
        V = disc * (p * V[:-1] + (1 - p) * V[1:])
        if american:
            Si = S * u ** (i - np.arange(i + 1)) * d ** np.arange(i + 1)
            V = np.maximum(V, (Si - K) if kind == 'call' else (K - Si))
    return float(V[0])


def merton_price(S, K, T, r, sigma, lam, mu_j, sig_j, kind='call', nmax=60):
    """Merton (1976): salturi log-normale ln(1+J) ~ N(mu_j, sig_j^2), intensitate lam; serie Poisson de preturi BS."""
    kappa = np.exp(mu_j + 0.5 * sig_j ** 2) - 1
    lam2 = lam * (1 + kappa)
    tot = 0.0
    for k in range(nmax):
        s_k = np.sqrt(sigma ** 2 + k * sig_j ** 2 / T)
        r_k = r - lam * kappa + k * np.log(1 + kappa) / T
        w = stats.poisson.pmf(k, lam2 * T)
        tot = tot + w * bs_price(S, K, T, r_k, s_k, kind)
    return tot


def heston_price(S, K, T, r, v0, kappa, theta, xi, rho, kind='call'):
    """Heston (1993): pretul call prin formula Lewis (2001), cu functia caracteristica a lui ln(S_T/F) (forma stabila)."""
    def phi(u):
        d = np.sqrt((rho * xi * 1j * u - kappa) ** 2 + xi ** 2 * (1j * u + u ** 2))
        g = (kappa - rho * xi * 1j * u - d) / (kappa - rho * xi * 1j * u + d)
        e = np.exp(-d * T)
        C = kappa * theta / xi ** 2 * ((kappa - rho * xi * 1j * u - d) * T - 2 * np.log((1 - g * e) / (1 - g)))
        D = (kappa - rho * xi * 1j * u - d) / xi ** 2 * (1 - e) / (1 - g * e)
        return np.exp(C + D * v0)
    x = np.log(S / K) + r * T
    f = lambda u: (np.exp(1j * u * x) * phi(u - 0.5j)).real / (u ** 2 + 0.25)
    call = S - np.sqrt(S * K) * np.exp(-r * T / 2) / np.pi * integrate.quad(f, 0, 200, limit=500, epsabs=1e-12)[0]
    return call if kind == 'call' else call - S + K * np.exp(-r * T)


def svi_w(k, a, b, rho, m, s):
    """SVI (Gatheral): varianta totala implicita w(k) = a + b (rho (k - m) + sqrt((k - m)^2 + s^2)), k = ln(K/F)."""
    return a + b * (rho * (k - m) + np.sqrt((k - m) ** 2 + s ** 2))


def svi_fit(k, w, weights=None):
    """Estimarea SVI prin cele mai mici patrate (ponderate), cu restrictii b >= 0, |rho| < 1, s > 0."""
    k, w = np.asarray(k), np.asarray(w)
    wt = np.ones_like(w) if weights is None else np.asarray(weights)
    def loss(p):
        return np.sum(wt * (svi_w(k, *p) - w) ** 2)
    best = None
    for rho0 in (-0.5, 0.0, 0.3):
        for s0 in (0.05, 0.2):
            x0 = [max(w.min() * 0.5, 1e-4), 0.1, rho0, 0.0, s0]
            res = optimize.minimize(loss, x0, method='L-BFGS-B',
                                    bounds=[(-1, 5), (1e-6, 10), (-0.999, 0.999), (-2, 2), (1e-4, 3)])
            if best is None or res.fun < best.fun:
                best = res
    a, b, rho, m, s = best.x
    return dict(a=a, b=b, rho=rho, m=m, s=s, rmse=np.sqrt(best.fun / wt.sum()),
                wmin=a + b * s * np.sqrt(1 - rho ** 2))


def variance_from_strip(K, Q, F, T, r=0.0):
    """Varianta implicita (formula VIX): 2/T sum dK/K^2 e^{rT} Q(K) - 1/T (F/K0 - 1)^2, Q = pretul OTM (mid)."""
    K, Q = np.asarray(K, float), np.asarray(Q, float)
    o = np.argsort(K); K, Q = K[o], Q[o]
    dK = np.empty_like(K)
    dK[1:-1] = (K[2:] - K[:-2]) / 2
    dK[0], dK[-1] = K[1] - K[0], K[-1] - K[-2]
    K0 = K[K <= F].max()
    return 2 / T * np.sum(dK / K ** 2 * np.exp(r * T) * Q) - (F / K0 - 1) ** 2 / T


def delta_hedge(paths, K, T, r, sigma_imp, n_rebal, kind='call'):
    """Vinde o optiune la pretul BS(sigma_imp) si acopera delta de n_rebal ori pana la scadenta.
    paths: matrice (n_traiectorii, n_pasi+1) de preturi pe o grila uniforma; n_pasi divizibil cu n_rebal.
    Intoarce eroarea de acoperire la scadenta (valoarea portofoliului de acoperire minus plata optiunii)."""
    npath, nstep = paths.shape[0], paths.shape[1] - 1
    every = nstep // n_rebal
    dt = T / nstep
    S0 = paths[:, 0]
    V0 = bs_price(S0, K, T, r, sigma_imp, kind)
    delta = bs_greeks(S0, K, T, r, sigma_imp, kind)['delta']
    cash = V0 - delta * S0
    for i in range(1, nstep + 1):
        cash = cash * np.exp(r * dt)
        if i % every == 0 and i < nstep:
            tau = T - i * dt
            d_new = bs_greeks(paths[:, i], K, tau, r, sigma_imp, kind)['delta']
            cash -= (d_new - delta) * paths[:, i]
            delta = d_new
    ST = paths[:, -1]
    payoff = np.maximum(ST - K, 0) if kind == 'call' else np.maximum(K - ST, 0)
    return cash + delta * ST - payoff

## Helper functions

In [ ]:
B_BOOT = 1000

A8_K = [80, 85, 90, 95, 100, 105, 110, 115, 120]

BFLY_EXPIRY = '2026-10-30 08:00:00'

def gbm_paths(S0, mu, sigma, T, nstep, npath, seed=SEED):
    rng = np.random.default_rng(seed)
    dt = T / nstep
    z = rng.standard_normal((npath, nstep))
    x = np.cumsum((mu - 0.5 * sigma ** 2) * dt + sigma * np.sqrt(dt) * z, axis=1)
    return S0 * np.exp(np.hstack([np.zeros((npath, 1)), x]))


def btc_surface():
    """SVI pe fiecare scadenta a lantului BTC (optiuni OTM cotate, cel putin 2 zile pana la scadenta)."""
    c = deribit_chain()
    t0 = c['snapshot_utc'].iloc[0]
    c['T'] = (c['expiry'] - t0).dt.total_seconds() / (365 * 86400)
    c = c[(c['T'] >= 2 / 365) & (c['mark_iv'] > 0) & (c['bid'] > 0)]
    c = c[((c['type'] == 'put') & (c['strike'] < c['forward'])) | ((c['type'] == 'call') & (c['strike'] >= c['forward']))].copy()
    c['k'] = np.log(c['strike'] / c['forward'])
    c['w'] = (c['mark_iv'] / 100) ** 2 * c['T']
    c = c[c['k'].abs() < 1.2]
    fits = {}
    for e, g in c.groupby('expiry'):
        if len(g) < 7:
            continue
        f = svi_fit(g['k'].values, g['w'].values)
        T = g['T'].iloc[0]
        iv = lambda k: 100 * np.sqrt(svi_w(k, f['a'], f['b'], f['rho'], f['m'], f['s']) / T)
        fits[e] = dict(T=T, days=T * 365, F=float(g['forward'].iloc[0]), n=int(len(g)), atm=float(iv(0.0)),
                       rr=float(iv(0.15) - iv(-0.15)), wing_dn=float(iv(-0.3)), wing_up=float(iv(0.3)),
                       rmse_iv=float(np.sqrt(np.mean((iv(g['k'].values) - g['mark_iv'].values) ** 2))), **f)
    tab = pd.DataFrame(fits).T
    tab.index.name = 'expiry'
    tab.to_csv(os.path.join(HERE, 'ch12_btc_svi.csv'))
    return c, tab, t0


def pick(tab, days):
    """Scadenta cea mai apropiata de un numar dat de zile."""
    return (tab['days'] - days).abs().astype(float).idxmin()


def vrp_sp500():
    """Prima de risc a variantei: VIX^2 minus varianta realizata in urmatoarele 21 de zile de tranzactionare."""
    px = pd.concat([load_close('sp500'), load_close('vix')], axis=1, join='inner').dropna()
    r = np.log(px['sp500']).diff()
    rv = 252 / 21 * (r ** 2).rolling(21).sum().shift(-21) * 1e4       # (%)^2 anualizat
    d = pd.DataFrame({'vix': px['vix'], 'iv2': px['vix'] ** 2, 'rv': rv, 'rv_past': rv.shift(21)}).dropna()
    d['vrp'] = d['iv2'] - d['rv']
    d['vrp_vol'] = d['vix'] - np.sqrt(d['rv'])
    return d


def vrp_btc():
    """Bitcoin: DVOL minus volatilitatea realizata in urmatoarele 30 de zile calendaristice."""
    dv = dvol_history()
    p = load_close('btc')
    r = np.log(p).diff()
    rv = 365 / 30 * (r ** 2).rolling(30).sum().shift(-30) * 1e4
    d = pd.concat([dv, rv.rename('rv')], axis=1, join='inner').dropna()
    d['vrp'] = d['dvol'] ** 2 - d['rv']
    d['vrp_vol'] = d['dvol'] - np.sqrt(d['rv'])
    return d


def delta_hedged_history():
    """S&P 500: vinde lunar o optiune call ATM pe 21 de zile la volatilitatea VIX si o acopera zilnic (1990-2026)."""
    px = pd.concat([load_close('sp500'), load_close('vix')], axis=1, join='inner').dropna()
    S, V = px['sp500'].values, px['vix'].values / 100
    dates = px.index
    H, T = 21, 21 / 252
    rows = []
    for i0 in range(0, len(px) - H, H):
        s0, sig = S[i0], V[i0]
        K = s0
        prem = float(bs_price(s0, K, T, 0.0, sig))
        delta = float(bs_greeks(s0, K, T, 0.0, sig)['delta'])
        cash = prem - delta * s0
        for i in range(1, H + 1):
            if i < H:
                d_new = float(bs_greeks(S[i0 + i], K, T * (H - i) / H, 0.0, sig)['delta'])
                cash -= (d_new - delta) * S[i0 + i]
                delta = d_new
        pnl = cash + delta * S[i0 + H] - max(S[i0 + H] - K, 0)
        rv = np.sqrt(252 / H * np.sum(np.diff(np.log(S[i0:i0 + H + 1])) ** 2))
        rows.append(dict(date=dates[i0], pnl=100 * pnl / s0, prem=100 * prem / s0, vix=sig * 100, rv=rv * 100))
    return pd.DataFrame(rows).set_index('date')


def rnd_from_svi(p, F, T, K):
    """Densitatea neutra la risc (Breeden-Litzenberger) din parametrii SVI, pe grila de preturi de exercitare K."""
    k = np.log(K / F)
    sig = np.sqrt(np.clip(svi_w(k, *p), 1e-10, None) / T)
    C = bs_price(F, K, T, 0.0, sig, 'call')
    return np.clip(np.gradient(np.gradient(C, K), K), 0, None)

## Analysis

In [ ]:
def a1_butterfly(h=1000.0, K0=85000.0):
    """Convexitatea in K pe lantul Bitcoin: fluture C(K-h) - 2C(K) + C(K+h) din preturile de marcare (USD) si costul
    executabil (aripile la ask, corpul la bid); densitatea aproximativa q(K) ~ fluture / h^2 (r = 0)."""
    c = deribit_chain()
    g = c[(c['expiry'] == pd.Timestamp(BFLY_EXPIRY)) & (c['type'] == 'call')].set_index('strike').sort_index()
    F = float(g['forward'].iloc[0])
    usd = lambda col: g[col] * F
    m, b, a = usd('mark_price'), usd('bid'), usd('ask')
    out = dict(F=F, h=h, K=K0, Cm=float(m[K0 - h]), C0=float(m[K0]), Cp=float(m[K0 + h]),
               bid0=float(b[K0]), askm=float(a[K0 - h]), askp=float(a[K0 + h]))
    out['bf_mark'] = out['Cm'] - 2 * out['C0'] + out['Cp']
    out['bf_exec'] = out['askm'] - 2 * out['bid0'] + out['askp']
    out['q'] = out['bf_mark'] / h ** 2
    out['mass'] = out['bf_mark'] / h                         # ~ P(K - h < S_T < K + h), ponderare triunghiulara
    # toate tripletele echidistante din lantul acestei scadente
    Ks = list(g.index)
    n_tr = n_neg_mark = n_neg_exec = 0
    neg, spr = [], []
    for i in range(1, len(Ks) - 1):
        for j in range(i + 1, len(Ks)):
            hh = Ks[j] - Ks[i]
            if Ks[i] - hh in g.index:
                n_tr += 1
                bm = m[Ks[i] - hh] - 2 * m[Ks[i]] + m[Ks[j]]
                n_neg_mark += int(bm < 0)
                if bm < 0:
                    neg.append(float(bm)); spr.append(float(a[Ks[i]] - b[Ks[i]]))
                if np.isfinite(a[Ks[i] - hh]) and np.isfinite(b[Ks[i]]) and np.isfinite(a[Ks[j]]) and b[Ks[i]] > 0:
                    n_neg_exec += int(a[Ks[i] - hh] - 2 * b[Ks[i]] + a[Ks[j]] < 0)
    out.update(n_triples=n_tr, n_neg_mark=n_neg_mark, n_neg_exec=n_neg_exec, worst_neg=float(min(neg)) if neg else 0.0,
               spread_neg=float(np.median(spr)) if spr else 0.0)
    return out


def a2_implied_forward():
    """Paritatea ca regresie: C - P = D F - D K pe toate preturile de exercitare cu call si put cotate (mid, USD)."""
    c = deribit_chain()
    g = c[(c['expiry'] == pd.Timestamp(BFLY_EXPIRY)) & (c['bid'] > 0) & (c['ask'] > 0)].copy()
    F0 = float(g['forward'].iloc[0])
    g['mid'] = 0.5 * (g['bid'] + g['ask']) * F0
    p = g.pivot_table(index='strike', columns='type', values='mid').dropna()
    y = p['call'] - p['put']
    X = sm.add_constant(pd.Series(p.index.values, index=p.index, name='K'))
    m = sm.OLS(y, X).fit(cov_type='HC1')
    D = -m.params['K']
    Fh = m.params['const'] / D
    # metoda delta: F = -a / b
    a_, b_ = m.params['const'], m.params['K']
    grad = np.array([-1 / b_, a_ / b_ ** 2])
    seF = float(np.sqrt(grad @ m.cov_params().values @ grad))
    return dict(n=int(len(p)), D=float(D), D_se=float(m.bse['K']), F=float(Fh), F_se=seF, F_exch=F0,
                kmin=float(p.index.min()), kmax=float(p.index.max()), r2=float(m.rsquared))


def a4_crr_error():
    """Eroarea CRR fata de Black-Scholes, inmultita cu N: oscilatie par/impar, ordinul 1/N."""
    S, K, T, r, s = 100.0, 100.0, 1.0, 0.05, 0.20
    bs = float(bs_price(S, K, T, r, s))
    return {str(N): dict(price=float(crr_price(S, K, T, r, s, N)), nerr=float(N * (crr_price(S, K, T, r, s, N) - bs)))
            for N in [50, 51, 100, 101, 200, 201, 400, 401]} | dict(bs=bs)


def a5_bs():
    """Black-Scholes pas cu pas: S = K = 100, T = 6 luni, r = 3%, sigma = 25%."""
    S, K, T, r, s = 100.0, 100.0, 0.5, 0.03, 0.25
    g = bs_greeks(S, K, T, r, s)
    return dict(d1=float(g['d1']), d2=float(g['d2']), Nd1=float(stats.norm.cdf(g['d1'])), Nd2=float(stats.norm.cdf(g['d2'])),
                pvk=K * np.exp(-r * T), C=float(bs_price(S, K, T, r, s)), P=float(bs_price(S, K, T, r, s, 'put')),
                delta=float(g['delta']), gamma=float(g['gamma']), vega=float(g['vega']), theta=float(g['theta']),
                nd1=float(stats.norm.pdf(g['d1'])))


def a6_delta_gamma():
    """Acoperire delta-gamma: vandut 1000 de call-uri A5; instrumente: call K = 110 (aceeasi scadenta) si actiunea."""
    S, T, r, s = 100.0, 0.5, 0.03, 0.25
    g1 = bs_greeks(S, 100, T, r, s); g2 = bs_greeks(S, 110, T, r, s)
    pos_delta, pos_gamma = -1000 * float(g1['delta']), -1000 * float(g1['gamma'])
    n2 = -pos_gamma / float(g2['gamma'])
    shares = -(pos_delta + n2 * float(g2['delta']))
    return dict(delta1=float(g1['delta']), gamma1=float(g1['gamma']), delta2=float(g2['delta']), gamma2=float(g2['gamma']),
                pos_delta=pos_delta, pos_gamma=pos_gamma, n2=n2, shares=shares, c2=float(bs_price(S, 110, T, r, s)),
                vega_left=-1000 * float(g1['vega']) + n2 * float(g2['vega']))


def a7_newton():
    """Volatilitatea implicita prin Newton: call S = K = 100, T = 3 luni, r = 2%, pret de piata 4.50, sigma_0 = 20%."""
    steps = newton_iv(4.50, 100.0, 100.0, 0.25, 0.02, sigma0=0.20, steps=3)
    return dict(steps=steps, exact=implied_vol(4.50, 100.0, 100.0, 0.25, 0.02))


def a8_lognormal_strip():
    """Banda de optiuni pentru o distributie log-normala (sigma = 20%, 90 de zile, F = 100, r = 0): valoarea exacta
    sigma^2 = 0.04; formula Cboe cu Delta K = 5 pe [80, 120]; grila densa pe [80, 120]; grila densa pe (0, infinit)."""
    F, T, s = 100.0, 90 / 365, 0.20
    q = lambda K: np.where(K < F, bs_price(F, K, T, 0.0, s, 'put'), bs_price(F, K, T, 0.0, s, 'call'))
    K5 = np.arange(80.0, 120.1, 5.0)
    Q5 = np.where(K5 == F, 0.5 * (bs_price(F, K5, T, 0.0, s, 'put') + bs_price(F, K5, T, 0.0, s, 'call')), q(K5))
    v5 = variance_from_strip(K5, Q5, F, T)
    Kd = np.linspace(80, 120, 40001)
    vd = 2 / T * integrate.trapezoid(q(Kd) / Kd ** 2, Kd)
    Kw = F * np.exp(np.linspace(-3, 3, 60001))
    vw = 2 / T * integrate.trapezoid(q(Kw) / Kw ** 2, Kw)
    return dict(exact=s ** 2, cboe=float(v5), dense=float(vd), wide=float(vw), vol_cboe=float(100 * np.sqrt(v5)),
                vol_dense=float(100 * np.sqrt(vd)), vol_wide=float(100 * np.sqrt(vw)),
                trunc=float(vd - s ** 2), disc=float(v5 - vd))


def b1_hedge_boot():
    """Eroarea de acoperire (abaterea standard) pentru N reechilibrari, cu interval bootstrap de 95%; panta log-log."""
    rng = np.random.default_rng(SEED)
    S0, K, T, r, s, mu = 100.0, 100.0, 0.25, 0.04, 0.20, 0.08
    paths = gbm_paths(S0, mu, s, T, 252, 10000, seed=SEED)
    prem = float(bs_price(S0, K, T, r, s))
    Ns = [6, 12, 21, 63, 126, 252]
    E = {n: delta_hedge(paths, K, T, r, s, n) for n in Ns}
    idx = rng.integers(0, paths.shape[0], size=(B_BOOT, paths.shape[0]))
    sd = {n: float(E[n].std()) for n in Ns}
    boot = {n: np.array([E[n][i].std() for i in idx]) for n in Ns}
    ci = {n: (float(np.quantile(boot[n], 0.025)), float(np.quantile(boot[n], 0.975))) for n in Ns}
    x = np.log(Ns)
    slope = float(np.polyfit(x, np.log([sd[n] for n in Ns]), 1)[0])
    bs_slopes = np.array([np.polyfit(x, np.log([boot[n][b] for n in Ns]), 1)[0] for b in range(B_BOOT)])
    fig, ax = plt.subplots(figsize=(6.2, 3.2))
    y = np.array([sd[n] for n in Ns]); lo = np.array([ci[n][0] for n in Ns]); hi = np.array([ci[n][1] for n in Ns])
    ax.errorbar(Ns, y, yerr=[y - lo, hi - y], fmt='o-', color=MainBlue, ms=4, capsize=3,
                label='Standard deviation of the hedging error, 95% bootstrap interval')
    ax.plot(Ns, y[0] * np.sqrt(Ns[0] / np.array(Ns)), color=Gray, lw=0.8, ls='--', label='Slope $-1/2$')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('Number of rebalancings $N$'); ax.set_ylabel('Standard deviation (per option)')
    legend_outside_bottom(ax, ncol=1, y=-0.2)
    save_fig('ch12_sem_hedge')
    return dict(prem=prem, sd=sd, ci=ci, slope=slope, slope_lo=float(np.quantile(bs_slopes, 0.025)),
                slope_hi=float(np.quantile(bs_slopes, 0.975)), ratio_63_252=sd[63] / sd[252])


def b2_delta_hedged():
    """Vanzatorul unei optiuni call pe o luna pe S&P 500, acoperita zilnic la VIX: media P&L, Newey-West, bootstrap pe blocuri."""
    d = delta_hedged_history()
    x = d['pnl'].values
    m, se = nw_mean(x, 3)
    rng = np.random.default_rng(SEED)
    n, L = len(x), 6
    means = []
    for _ in range(2000):
        st = rng.integers(0, n - L, size=int(np.ceil(n / L)))
        means.append(np.concatenate([x[s:s + L] for s in st])[:n].mean())
    worst5 = np.sort(x)[:5]
    return dict(n=n, mean=m, se=se, t=m / se, lo=float(np.quantile(means, 0.025)), hi=float(np.quantile(means, 0.975)),
                win=float((x > 0).mean()), mean_ex5=float(np.sort(x)[5:].mean()), worst5_sum=float(worst5.sum()),
                total=float(x.sum()), share_worst5=float(-worst5.sum() / x.sum()))


def b3_svi():
    """SVI pe scadenta BTC cea mai apropiata de 30 de zile; bootstrap pe perechi pentru volatilitatea ATM si asimetrie."""
    c, tab, t0 = btc_surface()
    e = pick(tab, 30)
    g = c[c['expiry'] == e]
    f = tab.loc[e]
    T = float(f['T'])
    k, w = g['k'].values, g['w'].values
    rng = np.random.default_rng(SEED)
    atm, rr, curves = [], [], []
    kk = np.linspace(k.min(), k.max(), 120)
    for _ in range(300):
        i = rng.integers(0, len(k), len(k))
        if len(np.unique(k[i])) < 6:
            continue
        p = svi_fit(k[i], w[i])
        iv = lambda x: 100 * np.sqrt(np.clip(svi_w(x, p['a'], p['b'], p['rho'], p['m'], p['s']), 1e-8, None) / T)
        atm.append(float(iv(0.0))); rr.append(float(iv(0.15) - iv(-0.15))); curves.append(iv(kk))
    curves = np.array(curves)
    # conditia de absenta a arbitrajului de tip fluture (Gatheral-Jacquier): g(k) >= 0
    a, b, rho, m, s = (float(f[x]) for x in ['a', 'b', 'rho', 'm', 's'])
    kg = np.linspace(-1.5, 1.5, 3001)
    W = svi_w(kg, a, b, rho, m, s)
    W1 = b * (rho + (kg - m) / np.sqrt((kg - m) ** 2 + s ** 2))
    W2 = b * s ** 2 / ((kg - m) ** 2 + s ** 2) ** 1.5
    gk = (1 - kg * W1 / (2 * W)) ** 2 - W1 ** 2 / 4 * (1 / W + 0.25) + W2 / 2
    fig, ax = plt.subplots(figsize=(6.6, 3.3))
    ax.fill_between(kk, np.quantile(curves, 0.025, axis=0), np.quantile(curves, 0.975, axis=0), color=LightGray,
                    label='95% bootstrap band of the SVI fit')
    ax.scatter(k, g['mark_iv'], s=10, color=IDAred, label='Quotes (mark implied volatility)', zorder=3)
    ax.plot(kk, 100 * np.sqrt(svi_w(kk, a, b, rho, m, s) / T), color=MainBlue, lw=1.4, label='SVI fit')
    ax.set_xlabel('Log-moneyness $k = \\ln(K/F)$'); ax.set_ylabel('Implied volatility (%)')
    ax.set_title(f"Bitcoin, expiry {pd.Timestamp(e).strftime('%d %b %Y')} ({f['days']:.0f} days)", fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=2, y=-0.2)
    save_fig('ch12_sem_svi')
    return dict(expiry=str(pd.Timestamp(e).date()), days=float(f['days']), n=int(len(k)), a=a, b=b, rho=rho, m=m, s=s,
                rmse=float(f['rmse_iv']), atm=float(f['atm']), rr=float(f['rr']),
                atm_lo=float(np.quantile(atm, 0.025)), atm_hi=float(np.quantile(atm, 0.975)),
                rr_lo=float(np.quantile(rr, 0.025)), rr_hi=float(np.quantile(rr, 0.975)), gmin=float(gk.min()),
                snapshot=str(t0))


def b5_vrp():
    """Prima de risc a variantei S&P 500: media, eroarea Newey-West (21 de lag-uri), subperioade."""
    d = vrp_sp500()
    out = {}
    for tag, x in [('all', d), ('p1', d.loc[:'2007-12-31']), ('p2', d.loc['2008-01-01':])]:
        m, se = nw_mean(x['vrp'], 21)
        mv, sev = nw_mean(x['vrp_vol'], 21)
        out[tag] = dict(n=int(len(x)), mean=m, se=se, t=m / se, lo=m - 1.96 * se, hi=m + 1.96 * se, mean_vol=mv, se_vol=sev,
                        share=float((x['vrp'] > 0).mean()), iv=float(x['vix'].mean()), rv=float(np.sqrt(x['rv']).mean()))
    naive = d['vrp'].std() / np.sqrt(len(d))
    out['naive_se'] = float(naive)
    out['ratio_se'] = out['all']['se'] / float(naive)
    return out


def b6_mz():
    """Mincer-Zarnowitz: varianta realizata pe urmatoarele 21 de zile pe VIX^2; test comun a = 0, b = 1 (HAC)."""
    d = vrp_sp500()
    m = sm.OLS(d['rv'], sm.add_constant(d[['iv2']])).fit(cov_type='HAC', cov_kwds={'maxlags': 21})
    w = m.wald_test('const = 0, iv2 = 1', scalar=True)
    m2 = sm.OLS(d['rv'], sm.add_constant(d[['iv2', 'rv_past']])).fit(cov_type='HAC', cov_kwds={'maxlags': 21})
    return dict(a=float(m.params['const']), a_se=float(m.bse['const']), b=float(m.params['iv2']), b_se=float(m.bse['iv2']),
                r2=float(m.rsquared), wald=float(w.statistic), wald_p=float(w.pvalue),
                b2=float(m2.params['iv2']), b2_se=float(m2.bse['iv2']), c2=float(m2.params['rv_past']),
                c2_se=float(m2.bse['rv_past']), r2_2=float(m2.rsquared))


def b7_vrp_btc():
    d = vrp_btc()
    m, se = nw_mean(d['vrp_vol'], 30)
    ratio = float(d['dvol'].mean() / np.sqrt(d['rv']).mean())
    s = vrp_sp500()
    return dict(n=int(len(d)), mean_vol=m, se_vol=se, t=m / se, share=float((d['vrp'] > 0).mean()), ratio=ratio,
                ratio_sp=float(s['vix'].mean() / np.sqrt(s['rv']).mean()),
                dvol=float(d['dvol'].mean()), rv=float(np.sqrt(d['rv']).mean()))


def b8_rnd_band(days=90, B=300):
    """Densitatea neutra la risc pentru scadenta BTC cea mai apropiata de 90 de zile: banda bootstrap pe perechi,
    probabilitatile P(S_T < 0.8F) si P(S_T > 1.2F) cu intervale, fata de densitatea log-normala ATM."""
    c, tab, t0 = btc_surface()
    e = pick(tab, days); f = tab.loc[e]
    F, T = float(f['F']), float(f['T'])
    p0 = [float(f[x]) for x in ['a', 'b', 'rho', 'm', 's']]
    K = F * np.exp(np.linspace(-1.5, 1.2, 2701))
    g = c[c['expiry'] == e]
    kq, wq = g['k'].values, g['w'].values
    def probs(q):
        A = integrate.trapezoid(q, K)
        lo = integrate.trapezoid(q[K <= 0.8 * F], K[K <= 0.8 * F]) / A
        hi = 1 - integrate.trapezoid(q[K <= 1.2 * F], K[K <= 1.2 * F]) / A
        return lo, hi
    q0 = rnd_from_svi(p0, F, T, K)
    p80, p120 = probs(q0)
    atm = np.sqrt(svi_w(0.0, *p0) / T)
    ln = lambda x: float(stats.lognorm.cdf(x * F, s=atm * np.sqrt(T), scale=F * np.exp(-0.5 * atm ** 2 * T)))
    rng = np.random.default_rng(SEED)
    P80, P120 = [], []
    while len(P80) < B:
        i = rng.integers(0, len(kq), len(kq))
        if len(np.unique(kq[i])) < 6:
            continue
        pb = svi_fit(kq[i], wq[i])
        a_, b_ = probs(rnd_from_svi([pb[x] for x in ['a', 'b', 'rho', 'm', 's']], F, T, K))
        P80.append(a_); P120.append(b_)
    return dict(expiry=str(pd.Timestamp(e).date()), days=float(f['days']), n=int(len(kq)), atm=float(100 * atm),
                p80=float(p80), p80_lo=float(np.quantile(P80, 0.025)), p80_hi=float(np.quantile(P80, 0.975)),
                p120=float(p120), p120_lo=float(np.quantile(P120, 0.025)), p120_hi=float(np.quantile(P120, 0.975)),
                ln80=ln(0.8), ln120=1 - ln(1.2), kf_min=float(np.exp(kq.min())), kf_max=float(np.exp(kq.max())))


def monthly_swap(iv, r, H, ppy):
    """P&L lunar al vanzatorului unui swap de varianta cu notional vega 1: (IV^2 - RV) / (2 IV), in puncte de volatilitate."""
    rows = []
    iv, r = iv.align(r, join='inner')
    for i0 in range(0, len(iv) - H, H):
        k = iv.iloc[i0]
        rv = ppy / H * np.sum(r.iloc[i0 + 1:i0 + H + 1].values ** 2) * 1e4
        rows.append(dict(date=iv.index[i0], iv=k, rv=np.sqrt(rv), pnl=(k ** 2 - rv) / (2 * k)))
    return pd.DataFrame(rows).set_index('date')


def perf(p, per_year=12):
    x = p['pnl']
    cum = x.cumsum()
    sr = x.mean() / x.std() * np.sqrt(per_year)
    rng = np.random.default_rng(SEED)
    n, L = len(x), 3
    srs = []
    xv = x.values
    for _ in range(2000):
        st = rng.integers(0, n - L, size=int(np.ceil(n / L)))
        y = np.concatenate([xv[s:s + L] for s in st])[:n]
        srs.append(y.mean() / y.std() * np.sqrt(per_year))
    return dict(n=int(n), mean=float(x.mean()), sd=float(x.std()), sharpe=float(sr), sr_lo=float(np.quantile(srs, 0.025)),
                sr_hi=float(np.quantile(srs, 0.975)), win=float((x > 0).mean()), worst=float(x.min()),
                worst_date=x.idxmin().strftime('%Y-%m'), maxdd=float((cum - cum.cummax()).min()), skew=float(stats.skew(x)),
                start=x.index[0].strftime('%Y-%m'), end=x.index[-1].strftime('%Y-%m'))


def c1_vrp_signal():
    px = pd.concat([load_close('sp500'), load_close('vix'), load_close('vix3m')], axis=1, join='inner')
    sp = pd.concat([load_close('sp500'), load_close('vix')], axis=1, join='inner').dropna()
    r_sp = np.log(sp['sp500']).diff().dropna()
    p_sp = monthly_swap(sp['vix'].loc[r_sp.index], r_sp, 21, 252)
    ratio = (px['vix'] / px['vix3m']).dropna()
    p_sp3 = p_sp.loc[ratio.index[0]:]
    cond = p_sp3[ratio.reindex(p_sp3.index) <= 1]
    dv = dvol_history()
    b = load_close('btc')
    r_b = np.log(b).diff().dropna()
    p_b = monthly_swap(dv, r_b, 30, 365)
    res = dict(sp=perf(p_sp), sp_since=perf(p_sp3), sp_cond=perf(cond), btc=perf(p_b),
               skipped=int(len(p_sp3) - len(cond)),
               skipped_mean=float(p_sp3.loc[~p_sp3.index.isin(cond.index), 'pnl'].mean()))
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
    axes[0].plot(p_sp.index, p_sp['pnl'].cumsum(), color=MainBlue, lw=1.1, label='S&P 500, every month (VIX)')
    axes[0].plot(cond.index, cond['pnl'].cumsum() + p_sp['pnl'].cumsum().reindex(cond.index).iloc[0] - cond['pnl'].iloc[0],
                 color=Forest, lw=1.0, ls='--', label='S&P 500, skip months that start in backwardation')
    axes[0].set_ylabel('Cumulative P&L (volatility points)')
    axes[0].axhline(0, color=Gray, lw=0.5)
    legend_outside_bottom(axes[0], ncol=1, y=-0.14)
    axes[1].plot(p_b.index, p_b['pnl'].cumsum(), color=Amber, lw=1.1, label='Bitcoin, every 30 days (DVOL)')
    axes[1].axhline(0, color=Gray, lw=0.5)
    legend_outside_bottom(axes[1], ncol=1, y=-0.14)
    plt.tight_layout()
    save_fig('ch12_sem_vrp_signal')
    return res

## Run

In [ ]:
for f in [a1_butterfly, a2_implied_forward, a4_crr_error, a5_bs, a6_delta_gamma, a7_newton, a8_lognormal_strip]:
    print(f.__name__, f())
print('B1', b1_hedge_boot())
print('B2', b2_delta_hedged())
print('B3', b3_svi())
print('B5', b5_vrp())
print('B6', b6_mz())
print('B7', b7_vrp_btc())
print('B8', b8_rnd_band())
print('C1', c1_vrp_signal())

![ch12_sem_hedge](./ch12_sem_hedge.png)

Output: `ch12_sem_hedge.pdf`

![ch12_sem_svi](./ch12_sem_svi.png)

Output: `ch12_sem_svi.pdf`

![ch12_sem_vrp_signal](./ch12_sem_vrp_signal.png)

Output: `ch12_sem_vrp_signal.pdf`